### MA(5/15),EMA(5/15) 변수 생성

In [1]:
import pandas as pd
import numpy as np

c:\Users\dddhs\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\dddhs\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [3]:
# =========================
# 1. 데이터 불러오기
# =========================
data_2008 = pd.read_csv(
    r'C:\Users\dddhs\TS_RL_proj\data\raw\data_2008.csv',
    index_col=0,
    parse_dates=True
)

data_2009 = pd.read_csv(
    r'C:\Users\dddhs\TS_RL_proj\data\raw\data_2009.csv',
    index_col=0,
    parse_dates=True
)


In [4]:
# 인덱스 정렬
data_2008 = data_2008.sort_index()
data_2009 = data_2009.sort_index()

# =========================
# 2. 기준 컬럼
# =========================
price_col = 'KOSPI 200'

# =========================
# 3. MA / EMA 생성
# =========================
data_2008['KOSPI 200_MA5'] = data_2008[price_col].rolling(window=5).mean()
data_2008['KOSPI 200_MA15'] = data_2008[price_col].rolling(window=15).mean()

data_2008['KOSPI 200_EMA5'] = data_2008[price_col].ewm(span=5, adjust=False).mean()
data_2008['KOSPI 200_EMA15'] = data_2008[price_col].ewm(span=15, adjust=False).mean()


In [5]:
# =========================
# 4. RSI(14) 생성
# =========================
delta = data_2008[price_col].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

# 단순이동평균 방식 RSI
avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()

rs = avg_gain / avg_loss
data_2008['KOSPI 200_RSI14'] = 100 - (100 / (1 + rs))

# 만약 loss가 0이면 RSI가 100에 가까워질 수 있음
# 필요시 아래처럼 무한대 처리 가능
data_2008['KOSPI 200_RSI14'] = data_2008['KOSPI 200_RSI14'].replace([np.inf, -np.inf], np.nan)


In [6]:
# =========================
# 5. Bollinger Bands(20, 2) 생성
# =========================
bb_mid = data_2008[price_col].rolling(window=20).mean()
bb_std = data_2008[price_col].rolling(window=20).std()

data_2008['KOSPI 200_BB_MID20'] = bb_mid
data_2008['KOSPI 200_BB_UPPER20'] = bb_mid + 2 * bb_std
data_2008['KOSPI 200_BB_LOWER20'] = bb_mid - 2 * bb_std

# =========================

In [7]:
# =========================
# 6. 2009 데이터에 붙일 피처 목록
# =========================
feature_cols = [
    'KOSPI 200_MA5',
    'KOSPI 200_MA15',
    'KOSPI 200_EMA5',
    'KOSPI 200_EMA15',
    'KOSPI 200_RSI14',
    'KOSPI 200_BB_MID20',
    'KOSPI 200_BB_UPPER20',
    'KOSPI 200_BB_LOWER20'
]

# =========================
# 7. data_2009 인덱스 기준으로 결합
# =========================
data_2009[feature_cols] = data_2008[feature_cols].reindex(data_2009.index)

# 백업
data_tech_features = data_2009.copy()

In [9]:
# =========================
# 8. 확인
# =========================
print(data_tech_features.head())
print("\n결측치 개수:")
print(data_tech_features[feature_cols].isna().sum())

# =========================

            Shanghai Comp     KODEX 200       TOPIX  Brent Crude Oil  USD/CNY  \
Date                                                                            
2009-04-17    2503.935059  12910.509766  679.428528        51.959999   6.8226   
2009-04-20    2557.456055  12992.271484  680.204834        51.959999   6.8230   
2009-04-21    2535.827881  12992.271484  663.898682        51.959999   6.8169   
2009-04-22    2461.345947  13166.940430  664.675171        51.959999   6.8195   
2009-04-23    2463.954102  13300.724609  669.333923        51.959999   6.8193   

             Gold Spot  KRW/JPY      KRW/USD      KOSDAQ   KOSPI 200  \
Date                                                                   
2009-04-17  867.400024   13.371  1325.800049  483.799988  171.330002   
2009-04-20  887.000000   13.536  1327.500000  491.940002  172.300003   
2009-04-21  882.099976   13.727  1354.300049  497.190002  171.960007   
2009-04-22  891.799988   13.726  1346.599976  509.899994  174.399994   


In [11]:
#data_tech_features 데이터의 USD/CNY 환율 컬럼을 삭제
data_tech_features = data_tech_features.drop(columns=['USD/CNY'])

In [ ]:
data_tech_features.to_csv(
    r'C:\Users\dddhs\TS_RL_proj\data\processed\data_tech_features.csv',
    index=True
)

AttributeError: module 'pandas' has no attribute 'to_csv'